# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidism/Machine-Learning-intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Task type: Ranking / Scoring**

This is a ranking (learning-to-rank / scoring) problem, not pure classification. The reviewer
doesn't need a binary yes/no on every page — they need an ordered list, because their real
constraint is limited review capacity (they can only look at the top 20-50 candidates per
cycle). What matters is getting the RELATIVE order right at the top of the list, not getting
every single page's label correct. That said, the underlying model I'll train is technically
a classifier (predicting probability of "needs refresh"), and I'll use its output probability
as the ranking score — so this is a scoring/ranking task built on top of a classification model.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Target: `trend_direction` (specifically, trend_direction == "down")**

This label comes from a **defined rule** built on measured change over time — it's derived by
comparing `impressions_last_30d` vs `impressions_prev_30d` (and similar clicks/sessions
comparisons) to classify each page's trajectory as "up," "stable," or "down." I'll treat
`trend_direction == "down"` as my binary target proxy: 1 if declining, 0 otherwise. This matters
for how I read results — the model learns to predict a rule-derived trend label, not an
independently verified ground-truth outcome like "this page was refreshed and traffic recovered."
It's a reasonable proxy for the decision I care about, but not the decision's actual outcome.

In [13]:
print(df["trend_direction"].value_counts())
print("\nProportion declining (trend_direction == 'down'):",
      (df["trend_direction"] == "down").mean().round(3))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Proportion declining (trend_direction == 'down'): 0.542


**Metric: Precision@50**

I'm defending Precision@50 because the reviewer's real constraint is capacity, not classifying
every page correctly. Precision@50 asks: "of the top 50 pages we'd tell someone to review, how
many were actually worth reviewing?" This directly matches the action (a reviewer working
through a ranked queue) and the cost structure from ML-02 (false negatives on high-impressions
pages are the costly mistake, so I care that the top of the list is dense with real candidates).
From the starter pipeline, baseline rules already scored 0.240 and a random forest scored 0.740
on this metric — giving me a concrete benchmark: "good" for my own model means beating 0.240 by
a meaningful margin, ideally approaching or exceeding what the reference model achieved.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
One row = one page

In [15]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/hamidism/Machine-Learning-intern/main/data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

# Show the unit of analysis: one row = one page
df.head(5)

Shape: (30000, 44)

Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


A fixed if-statement rule (e.g., "flag if days_since_update > 180 AND impressions_90d > 500")
only checks one or two thresholds in isolation. But from the ML-02 exploration, only 17 pages
(0.1%) actually matched that exact combination — meaning a simple rule misses almost everyone
who might still be worth reviewing, because decline shows up through many overlapping, weaker
signals at once (position drift, CTR mismatch for position, content age, engagement change)
rather than one clean threshold crossing. A model can weigh and combine dozens of such signals
simultaneously and learn non-obvious interactions between them — which is exactly why the
starter pipeline's learned model reached 0.740 Precision@50 versus the hand rule's 0.240. That
~3x gap is the clearest evidence that the real pattern here is too messy for a fixed rule to
capture well.

In [16]:
# Confirm how rare the simple rule-based combination actually is
simple_rule_flag = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
print("Pages matching simple fixed rule:", simple_rule_flag.sum(),
      f"({(simple_rule_flag.mean()*100):.1f}% of dataset)")

print("\nActual declining pages (trend_direction == 'down'):",
      (df["trend_direction"] == "down").sum(),
      f"({(df['trend_direction']=='down').mean()*100:.1f}% of dataset)")

overlap = (simple_rule_flag & (df["trend_direction"] == "down")).sum()
print(f"\nOf the {simple_rule_flag.sum()} rule-flagged pages, {overlap} are actually declining")

Pages matching simple fixed rule: 17 (0.1% of dataset)

Actual declining pages (trend_direction == 'down'): 16262 (54.2% of dataset)

Of the 17 rule-flagged pages, 16 are actually declining


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.